In [1]:
import os

HOME = "/Users/chandler/Downloads/CVCP"  # Set your project directory path

# Import torch FIRST before patching
import torch

# Patch torch.load to handle PyTorch 2.6+ weights_only changes
# This allows loading YOLO models without security errors
_original_torch_load = torch.load

def _patched_torch_load(f, map_location=None, pickle_module=None, weights_only=None, **kwargs):
    """Patched torch.load that forces weights_only=False for YOLO model compatibility"""
    return _original_torch_load(f, map_location=map_location, pickle_module=pickle_module, 
                                 weights_only=False, **kwargs)

# Apply the patch
torch.load = _patched_torch_load

# Now import ultralytics and other modules AFTER patching
import ultralytics
from ultralytics import YOLO # Import YOLO class. This class is used to create a YOLOv8 model
from IPython.display import display, Image
from roboflow import Roboflow
from tqdm import tqdm
from ultralytics.nn.tasks import DetectionModel
import torch.serialization
from torch.nn.modules.container import Sequential
import torch.nn as nn

print(f"Project directory: {HOME}")
print("PyTorch patched for YOLO model loading compatibility")
HOME

Project directory: /Users/chandler/Downloads/CVCP
PyTorch patched for YOLO model loading compatibility


'/Users/chandler/Downloads/CVCP'

_______________________________________________________________________________________________

_______________________________________________________________________________________________

# Training the model
modify the /data_path/ yourselve

modify data.yaml file as well

In [ ]:
%cd {HOME}
HOME
data_path= "/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml"
model = YOLO("yolov8n.yaml")
results = model.train(data= data_path, epochs=50, imgsz=640, plots=True)

#/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml

# OPTIONAL: Upgrade Ultralytics (Recommended)
If you encounter errors, run this cell to upgrade ultralytics to the latest version compatible with PyTorch 2.6+


In [ ]:
# Uncomment and run this if you get "ckpt_file" or other loading errors
# !pip install --upgrade ultralytics
# After upgrading, restart the kernel and re-run Cell 0


#Model Fine-Tuning


In [3]:
# Fine-tune YOLOv8n on the weed/crop dataset
# SIMPLIFIED VERSION - Direct fine-tuning without Ray Tune hyperparameter search
# This avoids subprocess issues with ultralytics 8.1.27 + PyTorch 2.6

%cd {HOME}

# Define paths - use absolute paths
model_path = '/Users/chandler/Downloads/CVCP/runs/detect/train/weights/best.pt'
data_path = '/Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml'

# Verify paths exist
import os
assert os.path.exists(model_path), f"Model not found at {model_path}"
assert os.path.exists(data_path), f"Data YAML not found at {data_path}"

print(f"Loading model from: {model_path}")
print(f"Using data config: {data_path}")
print(f"Ultralytics version: {ultralytics.__version__}")

# Load the trained model
print("\nLoading model...")
model = YOLO(str(model_path))
print("✓ Model loaded successfully!")

# Fine-tune the model with improved hyperparameters
# We'll use manually selected good hyperparameters instead of Ray Tune search
print("\n=== Starting Fine-tuning ===")
print("Training with improved hyperparameters...")

fine_tune_results = model.train(
    data=data_path,
    epochs=100,              # Extended training
    imgsz=640,               # Image size
    batch=16,                # Batch size (adjust based on your GPU memory)
    optimizer='AdamW',       # AdamW optimizer (better than SGD for fine-tuning)
    lr0=0.001,               # Lower initial learning rate for fine-tuning
    lrf=0.0001,              # Lower final learning rate
    momentum=0.937,          # Momentum
    weight_decay=0.0005,     # Weight decay
    warmup_epochs=5,         # Warmup epochs
    warmup_momentum=0.8,     # Warmup momentum
    box=7.5,                 # Box loss weight
    cls=0.5,                 # Classification loss weight
    dfl=1.5,                 # Distribution focal loss weight
    # Data augmentation
    hsv_h=0.015,             # HSV-Hue augmentation
    hsv_s=0.7,               # HSV-Saturation augmentation
    hsv_v=0.4,               # HSV-Value augmentation
    degrees=10.0,            # Rotation augmentation
    translate=0.1,           # Translation augmentation
    scale=0.5,               # Scale augmentation
    shear=2.0,               # Shear augmentation
    perspective=0.0,         # Perspective augmentation
    flipud=0.0,              # Flip up-down probability
    fliplr=0.5,              # Flip left-right probability
    mosaic=1.0,              # Mosaic augmentation probability
    mixup=0.1,               # Mixup augmentation probability
    # Monitoring
    plots=True,              # Generate plots
    save=True,               # Save checkpoints
    save_period=10,          # Save checkpoint every N epochs
    val=True,                # Validate during training
    # Output
    project='fine_tuning',   # Project directory
    name='yolov8n_finetuned', # Run name
    exist_ok=True,           # Overwrite if exists
    patience=50,             # Early stopping patience
    device=0,                # GPU device (use 'cpu' for CPU or 'mps' for Mac M1/M2)
)

print("\n=== Fine-tuning Complete ===")
print(f"Fine-tuned model saved at: {HOME}/fine_tuning/yolov8n_finetuned/weights/best.pt")

# Validate the fine-tuned model
print("\n=== Validating Fine-tuned Model ===")
validation_results = model.val()
print(f"\nFine-tuned mAP50: {validation_results.box.map50:.4f}")
print(f"Fine-tuned mAP50-95: {validation_results.box.map:.4f}")

# Compare with original model
print("\n=== Comparing with Original Model ===")
original_model = YOLO(model_path)
original_val = original_model.val(data=data_path)
print(f"Original mAP50: {original_val.box.map50:.4f}")
print(f"Original mAP50-95: {original_val.box.map:.4f}")

# Show improvement
map50_improvement = (validation_results.box.map50 - original_val.box.map50) * 100
map_improvement = (validation_results.box.map - original_val.box.map) * 100
print(f"\n📊 Improvement:")
print(f"  mAP50: {map50_improvement:+.2f}% {'📈' if map50_improvement > 0 else '📉'}")
print(f"  mAP50-95: {map_improvement:+.2f}% {'📈' if map_improvement > 0 else '📉'}")



This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.


/Users/chandler/Downloads/CVCP
Loading model from: /Users/chandler/Downloads/CVCP/runs/detect/train/weights/best.pt
Using data config: /Users/chandler/Downloads/CVCP/datasets_yolocp/weed-crop-aerial-2/data.yaml
Ultralytics version: 8.1.27

Loading model...
✓ Model loaded successfully!

=== Starting Fine-tuning ===
Training with improved hyperparameters...
New https://pypi.org/project/ultralytics/8.3.229 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.1.27 🚀 Python-3.10.18 torch-2.8.0 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


_______________________________________________________________________________________________

# Model Evaluation
When we are analysing how well YOLO is at predicting the contents of an image, there are several metrics we can use.
The most important ones are the **training loss** and the **validation loss**. The lower these values are, the better your algorithm is at predicting data. 

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

# Furthermore, here is the F-1 Curve
The F-1 curve tells us the overall performance of our model. It is particularly insightful because it **accounts for underrepresented classes**.
Imagine you have a thousand pictures of dogs and five of cats. You might have high accuracy if you always output dogs, but your F1 score will reflect this issue. 

In [ ]:
Image(filename=f'{HOME}/runs/detect/train/F1_curve.png', width=600)

_______________________________________________________________________________________________

## Testing the model
Previously, the model only saw pictures in the **train** folder. Now, we will show it the pictures in the **test** folder, pictures the model has never seen before. Based on how good the model's performance is with the test images, we can have an idea of what the model's performance with data in the real world will be.

## Test our model

In [ ]:
# Load a model
%cd {HOME}
model_path=f"{HOME}/runs/detect/train/weights/best.pt"
model_2 = YOLO(model_path)  # our trained YOLOv8n model

# Run batched inference on a list of images
results_2 = model_2(test1) 

# Process results list
for result in results_2:
    result.show()  # display to screen